<div class="alert alert-block alert-info">
Before you start running this notebook, create a cloud app in Verily Workbench with a <b>low compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab</b>.</li>
    <li>On the <b>Compute options</b> step, the default profile (2 CPUs, 13 GB of memory) is sufficient.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook starts the short-read structural variant summary for samples that have both long-read data and GATK-SV calls. It selects those ~989 participants, subsets the v7 GATK-SV VCF, and counts PASS sites and SVTYPEs as input to a formatted comparison table.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
# Dataset Builder query: long-read participants who also have short-read SV calls.
import pandas
import os

# This query represents dataset "Long reads with short read SVs, basic info" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_51778023_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_structural_variant_data = 1 ) )"""

dataset_51778023_person_df = pandas.read_gbq(
    dataset_51778023_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_51778023_person_df.head(5)

In [ ]:
# Write their IDs, one per line, for bcftools.
with open('sample_names.txt', 'w') as wf:
    for sample_name in list(dataset_51778023_person_df.person_id):
        wf.write(f'{sample_name}\n')

In [ ]:
# Controlled Tier short-read SV callset (GATK-SV), downloaded once (requester-pays bucket).
if not os.path.exists("AoU_srWGS_SV.vcf.gz"):
    !gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v7/wgs/short_read/structural_variants/vcf/AoU_srWGS_SV.vcf.gz* .

In [ ]:
# Check how many of the listed samples are in the VCF.
!bcftools view -S sample_names.txt AoU_srWGS_SV.vcf.gz | grep -m1 '^#CHROM' | sed 's/\t/\n/g' | tail -n +10 | wc -l

In [ ]:
# Subset to those samples, keep PASS sites only, and drop genotype columns (-G): the table counts sites.
!bcftools view -S sample_names.txt -f "PASS" -G -O z -o AoU_srWGS_SV.subset.vcf.gz AoU_srWGS_SV.vcf.gz

In [ ]:
!tabix -p vcf AoU_srWGS_SV.subset.vcf.gz

In [ ]:
# Number of PASS SV sites, then counts by ALT allele and by SV type (the ALT with angle brackets and any
# ':' subtype removed).
!bcftools view AoU_srWGS_SV.subset.vcf.gz | grep -v '^#' | wc -l

In [ ]:
!bcftools view AoU_srWGS_SV.subset.vcf.gz | grep -v '^#' | cut -f5 | sort | uniq -c

In [ ]:
!bcftools view AoU_srWGS_SV.subset.vcf.gz | grep -v '^#' | cut -f5 | sed 's/:.*//g' | sed 's/[<>]//g' | sort | uniq -c

<div class="alert alert-block alert-danger">
The remainder of this notebook is copied from our internal app.terra.bio production workspace. That data is queued for migration from internal storage to this public Researcher Workbench workspace. Please note that code below may not function properly until the migration is complete.
<br/>